# FACE RECOGNITION with 10 Algorithms

### FACE RECOGNITION using OpenCV2 and Ada Boost Learning Algorithm

In [ ]:
# Create_Database
import cv2, sys, numpy as np, os, time
count = 0
size = 4
database_dir = "database"
if not os.path.exists(database_dir):
    os.mkdir(database_dir)
fn_name = input("Enter the Person's Name: ")
path = os.path.join(database_dir, fn_name)
if not os.path.isdir(path):
    os.mkdir(path)
(im_width, im_height) = (112, 92)
haar_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')
webcam = cv2.VideoCapture(0)

print("-----------------------Taking pictures----------------------")
print("--------------------Give some expressions---------------------")

while count < 45:
    (rval, im) = webcam.read()
    im = cv2.flip(im, 1, 0)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    mini = cv2.resize(gray,(gray.shape[1]//size, gray.shape[0]//size))
    faces = haar_cascade.detectMultiScale(mini)
    faces = sorted(faces, key=lambda x: x[3])
    if faces:
        face_i = faces[0]
        (x, y, w, h) = [v * size for v in face_i]
        face = gray[y:y + h, x:x + w]
        face_resize = cv2.resize(face, (im_width, im_height))
        pin=sorted([int(n[:n.find('.')]) for n in os.listdir(path)
               if n[0]!='.' ]+[0])[-1] + 1
        cv2.imwrite('%s/%s.png' % (path, pin), face_resize)
        cv2.rectangle(im, (x, y), (x + w, y + h), (0, 255, 0), 3)
        cv2.putText(im, fn_name, (x - 10, y - 10), cv2.FONT_HERSHEY_PLAIN,
            1,(0, 255, 0))
        time.sleep(0.38)
        count += 1
    cv2.imshow('OpenCV', im)
    key = cv2.waitKey(10)
    if key == 27:
        break
print(str(count) + " images taken and saved to " + fn_name +" folder in database ")

In [ ]:
# def is_live(frame, gray):
#     eyes = eye_cascade.detectMultiScale(gray)
#     return len(eyes) >= 2 

# webcam = cv2.VideoCapture(0)
# liveness_frames_required = 1000
# live_count = 0

# print("Checking for liveness... please show your face to the camera")
# while live_count < liveness_frames_required:
#     ret, frame = webcam.read()
#     if not ret:
#         continue
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
#     if is_live(frame, gray):
#         live_count += 1
#         cv2.putText(frame, f"Liveness frames: {live_count}/{liveness_frames_required}",
#                     (10,30), cv2.FONT_HERSHEY_PLAIN, 1.5, (0,255,0), 2)
#     else:
#         cv2.putText(frame, "No face detected. Please show your face!",
#                     (10,30), cv2.FONT_HERSHEY_PLAIN, 1.5, (0,0,255), 2)
    
#     cv2.imshow("Liveness Check", frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         webcam.release()
#         cv2.destroyAllWindows()
#         exit()

# cv2.destroyAllWindows()
# print("Liveness confirmed Starting face recognition...")

In [ ]:

import dlib
from scipy.spatial import distance as dist

def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_pred

(lStart, lEnd) = (42, 48)
(rStart, rEnd) = (36, 42)

COUNTER = 0
TOTAL = 0
EYE_AR_THRESH = 0.3
EYE_AR_CONSEC_FRAMES = 3

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rects = detector(gray, 0)

    for rect in rects:
        shape = predictor(gray, rect)
        shape = [(shape.part(i).x, shape.part(i).y) for i in range(68)]

        leftEye = shape[lStart:lEnd]
        rightEye = shape[rStart:rEnd]

        leftEAR = eye_aspect_ratio(leftEye)
        rightEAR = eye_aspect_ratio(rightEye)

        ear = (leftEAR + rightEAR) / 2.0

        if ear < EYE_AR_THRESH:
            COUNTER += 1
        else:
            if COUNTER >= EYE_AR_CONSEC_FRAMES:
                TOTAL += 1
            COUNTER = 0

        cv2.putText(frame, f"Blinks: {TOTAL}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame, f"EAR: {ear:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Frame", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if TOTAL >= 5:
        print("10 blinks detected! Liveness confirmed.")
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2, sys, numpy, os

size = 4
haar_file = 'haarcascade_frontalface_default.xml'
datasets = 'database'

print('Training...')
# Create a list of images and a list of corresponding names
(images, lables, names, id) = ([], [], {}, 0)
for (subdirs, dirs, files) in os.walk(datasets):
    for subdir in dirs:
        names[id] = subdir
        subjectpath = os.path.join(datasets, subdir)
        for filename in os.listdir(subjectpath):
            path = subjectpath + '/' + filename
            lable = id
            images.append(cv2.imread(path, 0))
            lables.append(int(lable))
        id += 1
(width, height) = (130, 100)

# Create a Numpy array from the two lists above
(images, lables) = [numpy.array(lis) for lis in [images, lables]]

# OpenCV trains a model from the images
# NOTE FOR OpenCV2: remove '.face'
model = cv2.face.LBPHFaceRecognizer_create()
model.train(images, lables)

face_cascade = cv2.CascadeClassifier(haar_file)
webcam = cv2.VideoCapture(0)
while True:
    (_, im) = webcam.read()
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        cv2.rectangle(im, (x, y), (x + w, y + h), (255, 0, 0), 2)
        face = gray[y:y + h, x:x + w]
        face_resize = cv2.resize(face, (width, height))
        # Try to recognize the face
        prediction = model.predict(face_resize)
        cv2.rectangle(im, (x, y), (x + w, y + h), (0, 255, 0), 3)
        if prediction[1] < 100:
            cv2.putText(im, '%s - %.0f' % (names[prediction[0]], prediction[1]), (x - 10, y - 10),
                        cv2.FONT_HERSHEY_PLAIN, 1, (0, 255, 0))
        else:
            cv2.putText(im, 'not recognized', (x - 10, y - 10), cv2.FONT_HERSHEY_PLAIN, 1, (0, 255, 0))

    cv2.imshow('OpenCV', im)
    if cv2.waitKey(1) & 0xFF ==ord('q'):
        break
        
webcam.release()
cv2.destroyAllWindows()

In [ ]:
'''
    mannual free up resources
'''
webcam.release()
cv2.destroyAllWindows()